# 04 — Figure 2: spatial distribution of flares on the solar disk

Reproduces **Figure 2** of the paper: locations of every C-, M-, and X-class flare produced by AR 13663 and AR 13664 during May 2024, drawn on an orthographic projection of the visible solar disk.

* **Left panel:** AR 13663 (May 01 – May 10).
* **Right panel:** AR 13664 (May 01 – May 15).

Marker colour encodes the flare class (blue = C, green = M, red = X) and marker size scales monotonically with class.

Coordinates were extracted from the iSolSearch and NOAA/SWPC catalogs and curated by hand against the original flare list (`data/gannon_storm_2024_flares.csv`).

Output: `figures/solar_activity_ASR_final.jpg`.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import matplotlib.lines as mlines
import cartopy.crs as ccrs

FIG_DIR = Path('..').resolve() / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)


## Flare coordinates and classes

Two dictionaries hold the (Carrington-longitude, latitude) pair and the GOES class string for every event. They are imported verbatim from the original `mapa_flares_paper.ipynb` (final cartopy cell).


In [ ]:
FLARES_AR13663 = {'region': 'NOAA 13663', 'date': 'May 01 – May 10, 2024', 'coords': [(-29, 25), (-25, 24), (-24, 24), (-24, 24), (-21, 24), (-17, 25), (-17, 24), (-13, 24), (-13, 24), (-7, 25), (-6, 24), (-6, 25), (-6, 24), (-5, 25), (-4, 24), (3, 26), (6, 26), (8, 25), (3, 26), (9, 26), (8, 25), (8, 27), (9, 24), (12, 24), (15, 25), (13, 25), (13, 25), (14, 25), (16, 24), (14, 24), (14, 24), (15, 25), (20, 26), (21, 24), (18, 25), (18, 25), (22, 24), (22, 26), (20, 25), (21, 25), (23, 25), (24, 24), (24, 25), (27, 24), (27, 24), (32, 25), (29, 24), (28, 26), (30, 24), (30, 26), (36, 24), (37, 25), (35, 25), (38, 24), (31, 27), (40, 24), (36, 24), (37, 26), (43, 25), (37, 28), (44, 24), (43, 25), (38, 25), (45, 25), (45, 27), (52, 25), (54, 25), (53, 25), (56, 26), (55, 25), (60, 25), (59, 27), (65, 25), (65, 24), (65, 25), (88, 26)], 'flares': ['C1.7', 'C3.7', 'C1.9', 'C1.9', 'M1.0', 'C2.7', 'C4.3', 'C3.6', 'C1.9', 'C2.6', 'C2.4', 'X1.6', 'C6.0', 'C6.7', 'M4.4', 'C7.5', 'M1.6', 'C5.5', 'C3.1', 'M9.1', 'C8.5', 'M1.5', 'C4.5', 'C4.2', 'C3.8', 'C6.3', 'C9.1', 'C4.0', 'M1.3', 'C5.6', 'C6.0', 'C4.4', 'C4.7', 'M3.2', 'M8.4', 'C8.8', 'C8.4', 'C5.5', 'X1.3', 'M1.3', 'C8.4', 'M7.4', 'X1.3', 'M1.3', 'M2.2', 'C8.7', 'M1.3', 'C2.0', 'C6.1', 'C3.9', 'M1.6', 'C5.8', 'C8.8', 'M1.3', 'X4.5', 'M1.5', 'C6.5', 'C9.3', 'C4.4', 'C6.2', 'C4.3', 'C2.7', 'C3.7', 'C2.9', 'C9.4', 'M2.6', 'M5.1', 'C9.5', 'M1.5', 'M1.0', 'M1.0', 'M8.2', 'X1.0', 'M4.5', 'M1.8', 'M2.1', 'M4.1', 'C8.0']}

FLARES_AR13664 = {'region': 'NOAA 13664', 'date': 'May 01 – May 15, 2024', 'coords': [(-67, -16), (-62, -20), (-60, -21), (-61, -20), (-61, -20), (-59, -20), (-59, -21), (-53, -22), (-45, -21), (-43, -21), (-40, -22), (-22, -21), (-19, -21), (-24, -18), (-18, -21), (-18, -21), (-5, -18), (1, -24), (-7, -20), (11, -22), (11, -22), (12, -18), (17, -22), (15, -20), (15, -19), (17, -20), (21, -23), (23, -21), (23, -22), (27, -22), (30, -22), (24, -20), (39, -16), (31, -17), (32, -17), (28, -17), (36, -22), (36, -21), (35, -19), (34, -17), (43, -21), (39, -15), (36, -15), (59, -19), (48, -20), (44, -17), (43, -17), (51, -19), (48, -20), (44, -17), (58, -20), (56, -19), (60, -19), (59, -19), (50, -17), (63, -20), (58, -19), (66, -20), (68, -17), (68, -17), (70, -17), (74, -20), (76, -18), (75, -20), (82, -16), (76, -18), (81, -20), (82, -19), (83, -17), (82, -18), (75, -17), (84, -20), (81, -18), (89, -22), (81, -17), (89, -16), (89, -17), (89, -17), (89, -17), (89, -17), (89, -18), (89, -18)], 'flares': ['C1.6', 'C3.7', 'C6.3', 'C8.7', 'C4.0', 'M2.7', 'M2.7', 'C5.3', 'C8.0', 'C4.6', 'C5.8', 'M2.3', 'C7.5', 'M1.3', 'C6.4', 'M1.0', 'M1.3', 'M2.4', 'C8.4', 'M1.8', 'X1.0', 'C6.6', 'C6.6', 'M2.0', 'M1.7', 'X1.0', 'M9.7', 'M4.0', 'M4.5', 'M1.7', 'M2.3', 'X2.2', 'M3.1', 'M2.9', 'M3.7', 'X1.1', 'M1.3', 'C9.8', 'M1.3', 'X3.9', 'M2.2', 'M5.9', 'C6.9', 'C9.1', 'M1.1', 'M1.7', 'M2.0', 'M1.1', 'M1.9', 'X5.8', 'M3.1', 'M1.6', 'X1.5', 'M1.7', 'M8.8', 'M1.2', 'C9.4', 'M3.2', 'C9.2', 'M1.6', 'M1.0', 'M1.5', 'C7.0', 'X1.0', 'C6.4', 'C9.4', 'M1.2', 'C5.8', 'C5.0', 'C5.3', 'C5.3', 'C6.4', 'M1.2', 'M1.4', 'M6.6', 'M3.7', 'M1.5', 'X1.7', 'X1.2', 'C9.6', 'X8.7', 'X3.4']}


In [ ]:
def setup_solar_map_publication(ax, region):
    """Draw one orthographic solar-disk panel with N/S/E/W labels,
    a circular boundary, and the flare markers for a single AR."""
    FONT_AXIS, FONT_TITLE = 22, 26
    data_transform = ccrs.PlateCarree()

    # Grid
    ax.set_global()
    gl = ax.gridlines(crs=data_transform, draw_labels=False,
                      linewidth=1.5, color='gray', alpha=0.5, linestyle='--')
    gl.xlocator = plt.FixedLocator(np.arange(-90, 91, 30))
    gl.ylocator = plt.FixedLocator(np.arange(-90, 91, 30))

    # Circular crop
    theta = np.linspace(0, 2 * np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    ax.set_boundary(mpath.Path(verts * radius + center), transform=ax.transAxes)

    # Cardinal labels
    offset = 0.05
    for txt, xy, ha, va in [
        ('N', (0.5, 1.0 + offset / 2), 'center', 'bottom'),
        ('S', (0.5, -offset),         'center', 'top'),
        ('W', (1.0 + offset, 0.5),    'left',   'center'),
        ('E', (-offset, 0.5),         'right',  'center'),
    ]:
        ax.text(*xy, txt, ha=ha, va=va, transform=ax.transAxes,
                fontsize=FONT_AXIS, fontweight='bold', family='sans-serif')

    # Inner titles
    title_box = dict(facecolor='white', alpha=0.85, edgecolor='none',
                     boxstyle='round,pad=0.3')
    ax.text(0.03, 0.94, region['region'], ha='left', va='top',
            transform=ax.transAxes, fontsize=FONT_TITLE,
            fontweight='bold', bbox=title_box)
    ax.text(0.97, 0.94, region['date'], ha='right', va='top',
            transform=ax.transAxes, fontsize=FONT_TITLE, bbox=title_box)

    # Flare markers
    size_by_class  = {'C': 12, 'M': 18, 'X': 26}
    color_by_class = {'C': 'blue', 'M': 'green', 'X': 'red'}
    n = min(len(region['coords']), len(region['flares']))
    for (lon, lat), flare in zip(region['coords'][:n], region['flares'][:n]):
        cls = flare[0]
        ax.plot(lon, lat, marker='o',
                markersize=size_by_class.get(cls, 8),
                color=color_by_class.get(cls, 'black'),
                alpha=0.8, markeredgecolor='white', markeredgewidth=1.5,
                transform=data_transform)


In [ ]:
geo_proj = ccrs.Orthographic(central_longitude=0, central_latitude=0)
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(18, 9),
                        subplot_kw={'projection': geo_proj})

for ax, region in zip(axs, [FLARES_AR13663, FLARES_AR13664]):
    setup_solar_map_publication(ax, region)

# Bottom legend
legend_elements = [
    mlines.Line2D([], [], color='white', marker='o', markerfacecolor='blue',
                  markersize=18, label='Class C',
                  markeredgecolor='white', markeredgewidth=1),
    mlines.Line2D([], [], color='white', marker='o', markerfacecolor='green',
                  markersize=24, label='Class M',
                  markeredgecolor='white', markeredgewidth=1),
    mlines.Line2D([], [], color='white', marker='o', markerfacecolor='red',
                  markersize=30, label='Class X',
                  markeredgecolor='white', markeredgewidth=1),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=24, frameon=False, bbox_to_anchor=(0.5, 0.0))

plt.tight_layout()
plt.subplots_adjust(bottom=0.15, wspace=0.1)
plt.savefig(FIG_DIR / 'solar_activity_ASR_final.jpg',
            format='jpg', dpi=300, bbox_inches='tight')
plt.show()
